# Metorial + Microsoft Semantic Kernel Example

This notebook demonstrates how to use Metorial tools with Microsoft's Semantic Kernel framework.

In [ ]:
# Install dependencies
%pip install metorial semantic-kernel python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("METORIAL_API_KEY"), "Set METORIAL_API_KEY"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.getenv("EXA_DEPLOYMENT_ID"), "Set EXA_DEPLOYMENT_ID"

In [ ]:
import semantic_kernel as sk
from semantic_kernel.connectors.ai.function_choice_behavior import (
  FunctionChoiceBehavior,
)
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents.chat_history import ChatHistory

from metorial import Metorial
from metorial.integrations.semantic_kernel import register_metorial_plugin

In [ ]:
metorial = Metorial(api_key=os.getenv("METORIAL_API_KEY"))

In [ ]:
async def run_kernel(query: str):
  async with metorial.provider_session(
    provider="openai",
    server_deployments=[os.getenv("EXA_DEPLOYMENT_ID")],
  ) as session:
    kernel = sk.Kernel()

    service = OpenAIChatCompletion(
      service_id="chat",
      ai_model_id="gpt-4o",
    )
    kernel.add_service(service)

    register_metorial_plugin(kernel, session)
    print("Metorial plugin registered")

    settings = kernel.get_prompt_execution_settings_from_service_id("chat")
    settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

    history = ChatHistory()
    history.add_user_message(query)

    result = await kernel.invoke_prompt(
      prompt="{{$history}}",
      history=history,
      settings=settings,
    )
    return result

In [ ]:
result = await run_kernel("Search for the latest Python 3.13 features")
print(result)